<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



*Build the ranked queue. Every row gets an archetype, an action, and a reason code.*

**Archetype -> action mapping**, built around this model's *actual* top permutation/importance signals from w05 (`days_with_impressions`, `impressions_90d`, `content_age_days`), not a generic staleness narrative:

| Archetype | Condition | Action |
|---|---|---|
| Fix CTR | Position <= 20, CTR below the position band's median | **Fix CTR** |
| Refresh (visibility-driven) | Low `days_with_impressions` relative to `content_age_days` (visibility didn't accumulate the way the page's age would suggest) AND stale (`days_since_last_update >= 91`) | **Refresh** -- this uses w05's real top feature, not an assumed staleness window |
| Support | Position 11-20, competitive CTR | **Support** |
| Monitor (deep) | Position > 20 | **Monitor** |
| Monitor (no flag) | Visible, no archetype condition met | **Monitor** |

Each archetype's actual decline rate is checked against the base rate below -- not assumed to work, per the same honesty check used in the previous version of this notebook.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

RANDOM_SEED = 42
TEST_SIZE = 0.2  # matches w05_model.ipynb
VISIBILITY_FLOOR = 500

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
n_total = len(df)

def build_features(df, train_idx_for_median):
    work = df.copy()
    work["has_position_data"] = (work["avg_position"] != 0).astype(int)
    work["avg_position"] = work["avg_position"].replace(0, np.nan)

    raw_numeric = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
                   "scroll_rate", "days_since_last_update", "content_age_days",
                   "word_count", "days_with_impressions"]
    numeric_feats = ["has_position_data"]
    for col in raw_numeric:
        if work[col].isnull().any():
            work[f"has_{col}"] = work[col].notna().astype(int)
            train_median = work.iloc[train_idx_for_median][col].median()
            work[f"{col}_filled"] = work[col].fillna(train_median)
            numeric_feats += [f"{col}_filled", f"has_{col}"]
        else:
            numeric_feats.append(col)

    freshness_dummies = pd.get_dummies(work["freshness_tier"], prefix="freshness")
    X_all = pd.concat([work[numeric_feats], freshness_dummies], axis=1)
    y_all = (work["trend_direction"] == "down").astype(int)
    return X_all, y_all, work

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

X_all, y_all, work = build_features(df, train_idx)
scaler = StandardScaler().fit(X_all.iloc[train_idx])
X_all_scaled = scaler.transform(X_all)

clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
clf.fit(X_all_scaled[train_idx], y_all.iloc[train_idx])

df["model_score"] = clf.predict_proba(X_all_scaled)[:, 1]

eligible = df["impressions_90d"] >= VISIBILITY_FLOOR
print(f"Visibility floor: impressions_90d >= {VISIBILITY_FLOOR}")
print(f"  Eligible: {eligible.sum()} of {n_total} ({eligible.mean():.1%})")

# --- Archetype logic, built around days_with_impressions (w05's real top feature) ---
position_valid = df["avg_position"] != 0
band_1_20 = position_valid & (df["avg_position"] <= 20)
median_ctr_by_band = df.loc[eligible & band_1_20, "ctr"].median()
median_visibility_ratio = (df.loc[eligible, "days_with_impressions"] / df.loc[eligible, "content_age_days"].clip(lower=1)).median()

def classify(row):
    if row["avg_position"] != 0 and row["avg_position"] <= 20 and row["ctr"] < median_ctr_by_band:
        return "fix_ctr", "Fix CTR"
    visibility_ratio = row["days_with_impressions"] / max(row["content_age_days"], 1)
    if visibility_ratio < median_visibility_ratio and row["days_since_last_update"] >= 91:
        return "low_visibility_ratio_stale", "Refresh"
    if row["avg_position"] != 0 and 11 <= row["avg_position"] <= 20 and row["ctr"] >= median_ctr_by_band:
        return "support_near_page1", "Support"
    if row["avg_position"] != 0 and row["avg_position"] > 20:
        return "deep_position_monitor", "Monitor"
    return "no_flag_monitor", "Monitor"

queue = df[eligible].copy()
classified = queue.apply(classify, axis=1, result_type="expand")
queue["reason_code"] = classified[0]
queue["action"] = classified[1]
queue_ranked = queue.sort_values("model_score", ascending=False).reset_index(drop=True)

archetype_summary = queue_ranked.groupby("action").agg(
    n=("trend_direction", "size"),
    observed_decline_rate=("trend_direction", lambda s: (s == "down").mean())
).round(3)
print("\nArchetype counts and observed decline rate:")
print(archetype_summary)

queue_base_rate = (queue_ranked["trend_direction"] == "down").mean()
print(f"\nEligible-population base rate: {queue_base_rate:.3f}")
for a in [x for x in archetype_summary.index if x != "Monitor"]:
    rate, n_a = archetype_summary.loc[a, "observed_decline_rate"], archetype_summary.loc[a, "n"]
    status = "WARNING: at/below base rate" if rate <= queue_base_rate else "OK: elevated vs base rate"
    print(f"{status} -- '{a}' (n={n_a}): {rate:.3f}")

print(f"\nTop 10 rows:")
print(queue_ranked[["content_id", "model_score", "action", "reason_code"]].head(10).to_string(index=False))

Visibility floor: impressions_90d >= 500
  Eligible: 16726 of 30000 (55.8%)

Archetype counts and observed decline rate:
            n  observed_decline_rate
action                              
Fix CTR  5893                  0.665
Monitor  7017                  0.554
Refresh  2445                  0.562
Support  1371                  0.573

Eligible-population base rate: 0.596
OK: elevated vs base rate -- 'Fix CTR' (n=5893): 0.665

Top 10 rows:
          content_id  model_score  action     reason_code
content_8f3b70ede1bc     0.869499 Fix CTR         fix_ctr
content_8ede62882d0b     0.867625 Fix CTR         fix_ctr
content_b08562686d22     0.865670 Fix CTR         fix_ctr
content_67a766790dd2     0.848664 Fix CTR         fix_ctr
content_26d48a980581     0.844939 Fix CTR         fix_ctr
content_2bc3b7c8b3d9     0.842821 Monitor no_flag_monitor
content_c94a53e3bfb8     0.842707 Monitor no_flag_monitor
content_3ff647c911d0     0.834366 Monitor no_flag_monitor
content_0361d8df96e6     0.8

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*



*What is this for? What can it not tell anyone? Use claim-ladder language throughout.*

**Intended use.** Decision support for an editor deciding what to open first -- not an automated action system. Per `writing-honest-claims/SKILL.md`'s claim ladder: *"these pages look worth reviewing first, because they are observed to share properties associated with decline in this dataset"* -- not that reviewing them will fix anything.

**What this can say:**
- The model **ranks pages at a measured precision@K**, validated out-of-sample under the same client-grouped split w05 used (Section 3 below reproduces the exact numbers). Base rate is always stated alongside it.
- Pages in an archetype **are associated with** a higher observed decline rate than the eligible-population average -- checked directly in Section 1, not assumed.

**What this cannot say:** that acting on a flagged page will recover its traffic (no causal design exists); that the model predicts search-engine ranking behavior; anything about an individual client's effort.

**Named limits carried from w06's audit:**
- Visibility floor of {VISIBILITY_FLOOR} impressions/90d, disclosed, not hidden.
- w05's own model comparison shows Random Forest underperforming at low K -- this playbook deliberately does NOT use RF for that reason, and that choice is stated here rather than left implicit.
- Split variance: precision@K varies across client-grouped splits on this 32-client panel (quantified in `w06_validation_audit.ipynb`).

In [2]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y_test = y_all.iloc[test_idx]
proba_test = clf.predict_proba(X_all_scaled[test_idx])[:, 1]
base_rate = y_test.mean()

print("=== Measured precision, Logistic Regression, matching w05's split exactly ===")
print(f"Base rate: {base_rate:.3f}")
results = {}
for k in [20, 50, 100]:
    p = precision_at_k(proba_test, y_test.values, k)
    results[k] = p
    print(f"Precision@{k}: {p:.3f}  (lift over base rate: {p - base_rate:+.3f})")

traffic_tiers = pd.qcut(df.loc[test_idx, "impressions_90d"], q=4, labels=["low","mid","high","top"], duplicates="drop")
test_frame = pd.DataFrame({"tier": traffic_tiers.values, "y_true": y_test.values, "proba": proba_test})
test_frame["pred"] = (test_frame["proba"] >= 0.5).astype(int)
test_frame["correct"] = (test_frame["pred"] == test_frame["y_true"]).astype(int)
print("\n=== Accuracy by traffic tier (n shown) ===")
print(test_frame.groupby("tier", observed=True)["correct"].agg(["mean", "count"]))


=== Measured precision, Logistic Regression, matching w05's split exactly ===
Base rate: 0.511
Precision@20: 0.750  (lift over base rate: +0.239)
Precision@50: 0.740  (lift over base rate: +0.229)
Precision@100: 0.730  (lift over base rate: +0.219)

=== Accuracy by traffic tier (n shown) ===
          mean  count
tier                 
low   0.554201   1559
mid   0.590164   1525
high  0.587126   1538
top   0.487995   1541


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



*Why does a person stay in the loop? What must never be automated?*

**Cost asymmetry:** a false positive costs an editor bounded, visible time; a false negative is invisible and compounds. This is why the system ranks and a person acts, not a claim about ranking quality.

**What must NOT be automated:**
- No automated content edits -- the system ranks, a person writes.
- No automated pruning or deletion -- below-floor means "not measurable," never "delete."
- No client-facing forecasts -- only past-tense, observed-pattern language.
- No automatic handling of the weakest-measured tier from Section 2.
- No performance management -- scores reflect search-demand movement, not effort.
- No unreviewed scoring of a client outside this validated panel.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



*Light, practical triggers — not a full MLOps system.*

1. **Precision drift trigger** -- if a fresh sample's precision@50 falls below `mean - 1sd` from `w06_validation_audit.ipynb`'s split history (computed below).
2. **Base-rate drift trigger** -- if the observed decline base rate moves >10 points from this playbook's build-time value.
3. **Client-mix trigger** -- re-run the grouped validation if the client roster changes meaningfully.
4. **Calendar cadence** -- quarterly minimum, per `flyrank-data/SKILL.md`'s panel-drift warnings.
5. **Model-choice health check** -- if a future re-run shows Random Forest overtaking Logistic Regression on this dataset, re-validate before switching -- don't assume the paper's model choice applies here just because it worked on a different dataset.

In [3]:
split_history_path = "work/outputs/split_before_after.csv"
if os.path.exists(split_history_path):
    split_hist = pd.read_csv(split_history_path)
    grouped_mean = split_hist["after_grouped_split"].mean()
    grouped_sd = split_hist["after_grouped_split"].std()
    trigger_threshold = grouped_mean - grouped_sd
    print(f"From w06: mean={grouped_mean:.3f}, sd={grouped_sd:.3f}, trigger threshold={trigger_threshold:.3f}")
else:
    grouped_mean = grouped_sd = trigger_threshold = None
    print("w06's split history not found -- run w06_validation_audit.ipynb first for a concrete threshold.")

print(f"\nBase rate this playbook was built on: {base_rate:.3f}")

w06's split history not found -- run w06_validation_audit.ipynb first for a concrete threshold.

Base rate this playbook was built on: 0.511


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

export_cols = [c for c in queue_ranked.columns if c != "client_id"]
assert "client_id" not in export_cols
queue_ranked[export_cols].to_csv("work/outputs/content_action_queue.csv", index=False)
print(f"Wrote {len(queue_ranked)} rows to work/outputs/content_action_queue.csv")

metrics = {
    "model": "logistic_regression",
    "model_choice_reason": "w05_model.ipynb showed LR beating base rate and the rule baseline at every K; Random Forest underperformed base rate at K=20/50",
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "visibility_floor_impressions_90d": VISIBILITY_FLOOR,
    "n_total_rows": int(n_total),
    "n_eligible_for_queue": int(eligible.sum()),
    "base_rate_test_split": round(float(base_rate), 4),
    "precision_at_k": {str(k): round(float(v), 4) for k, v in results.items()},
    "archetype_counts": archetype_summary["n"].to_dict(),
    "archetype_observed_decline_rate": archetype_summary["observed_decline_rate"].to_dict(),
    "grouped_split_mean_precision_at_50": None if grouped_mean is None else round(float(grouped_mean), 4),
    "grouped_split_sd_precision_at_50": None if grouped_sd is None else round(float(grouped_sd), 4),
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/playbook_metrics.json")

fig, ax = plt.subplots(figsize=(7, 4.5))
s = archetype_summary.sort_values("n", ascending=False)
ax.bar(s.index, s["observed_decline_rate"], color="#2a78d6")
for i, (n_val, rate) in enumerate(zip(s["n"], s["observed_decline_rate"])):
    ax.text(i, rate + 0.01, f"n={n_val}", ha="center", fontsize=9)
ax.axhline(base_rate, color="gray", linestyle="--", linewidth=1, label=f"base rate ({base_rate:.2f})")
ax.set_ylabel("Observed decline rate")
ax.set_title("Observed decline rate by archetype (Logistic Regression scoring)")
ax.legend()
plt.tight_layout()
plt.savefig("work/figures/archetype_decline_rate.png", dpi=150)
plt.close()
print("Wrote work/figures/archetype_decline_rate.png")


Wrote 16726 rows to work/outputs/content_action_queue.csv
Wrote work/outputs/playbook_metrics.json
Wrote work/figures/archetype_decline_rate.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [5]:
import os, json
import pandas as pd

assert os.path.exists("work/outputs/content_action_queue.csv")
assert os.path.exists("work/outputs/playbook_metrics.json")
assert os.path.exists("work/figures/archetype_decline_rate.png")

exported = pd.read_csv("work/outputs/content_action_queue.csv")
assert "client_id" not in exported.columns
assert {"content_id", "model_score", "action", "reason_code"}.issubset(exported.columns)

with open("work/outputs/playbook_metrics.json") as f:
    m = json.load(f)
assert m["model"] == "logistic_regression", "Playbook should be scored with the validated winner, LR."

print("Self-check PASSED.")
print(f"Model: {m['model']}  |  base_rate: {m['base_rate_test_split']}  |  precision@50: {m['precision_at_k']['50']}")

Self-check PASSED.
Model: logistic_regression  |  base_rate: 0.511  |  precision@50: 0.74
